In [0]:
result = spark.sql("""
  SELECT COUNT(*) AS failed_count
    FROM tibia_analytics.bronze.raw_characters
   WHERE source_file_date = (
           SELECT MAX(source_file_date)
             FROM tibia_analytics.bronze.raw_characters
         )
     AND (
           character.character.name IS NULL
           OR TRIM(character.character.name) = ''
           OR character.character.world IS NULL
           OR TRIM(character.character.world) = ''
         )
""")
row = result.first()
missing_character_identity_count = row["failed_count"] if row is not None else 0

In [0]:
check_results = {
    "Characters with missing identity": missing_character_identity_count
}

failed_checks = [
    f"{check_name} ({failed_count} occurrence(s))"
    for check_name, failed_count in check_results.items()
    if failed_count > 0
]

if failed_checks:
    raise Exception(
        "Bronze Characters data quality checks failed:\n- "
        + "\n- ".join(failed_checks)
    )

print("All Bronze Characters data quality checks passed.")